# 03 · Evidence-score regression for known drug–gene associations

This notebook retains the project's target-prioritisation concept. It learns a **constructed composite evidence score**, not drug efficacy, sensitivity, binding affinity, or new interactions. The transparent composite ranking from notebook 02 remains the reference.

**Input:** `results/tables/target_prioritisation/<contrast>/dgi_edges_gnn.csv` plus its completed `provenance.json`.
**Outputs:** contrast-specific tables, models, figures and report under `evidence_score_regression/<contrast>/`.

For Colab, copy/clone the project including `scripts/utils/evidence_gnn_functions.py` and both input files. Install PyTorch, PyTorch Geometric, pandas, scikit-learn and matplotlib in that environment. Set `REPO_ROOT` below to the copied project's root. A CSV alone is insufficient.


In [1]:
from pathlib import Path
import sys, json, hashlib, platform
from datetime import datetime, timezone
import importlib.util
import numpy as np
import pandas as pd
import torch

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / 'scripts/utils/evidence_gnn_functions.py').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Set REPO_ROOT to the HCC_DD project directory, including scripts/.')
# Load this helper directly to avoid unrelated single-cell/R dependencies in utils.__init__.
spec = importlib.util.spec_from_file_location('hcc_evidence_gnn', REPO_ROOT / 'scripts/utils/evidence_gnn_functions.py')
gnn = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = gnn
spec.loader.exec_module(gnn)

CONTRAST = 'pooled'  # e.g. celltype_T_cell; must match the completed notebook 02 run
TARGET_RUN_NAME = 'target_prioritisation'
RUN_NAME = 'evidence_score_regression'
INPUT_DIR = REPO_ROOT / 'results/tables' / TARGET_RUN_NAME / CONTRAST
EDGE_FILE = INPUT_DIR / 'dgi_edges_gnn.csv'
TABLES_DIR = REPO_ROOT / 'results/tables' / RUN_NAME / CONTRAST
FIGURES_DIR = REPO_ROOT / 'results/figures' / RUN_NAME / CONTRAST
REPORTS_DIR = REPO_ROOT / 'results/reports' / RUN_NAME / CONTRAST
MODELS_DIR = REPO_ROOT / 'models' / RUN_NAME / CONTRAST

SPLIT_SEED = 42
INITIALIZATION_SEEDS = (42, 43, 44)
TRAINING = dict(hidden=64, embed=32, dropout=.2, lr=.003, weight_decay=1e-4,
                epochs=300, patience=40)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(4)
print('Device:', DEVICE, '| contrast:', CONTRAST)


Device: cpu | contrast: pooled


## Validate the current input and prepare features

The split targets approximately 70/15/15 of **gene × candidate drug alias groups**; row proportions can differ. All same-gene pairs connected by identical names or database IDs are kept in one split. Names do not prove chemical equivalence, so database identities stay distinct unless confirmed upstream. The full unweighted graph of known associations remains visible, including validation and test connections; this is explicitly a transductive known-association task. It cannot evaluate recovery of unknown drug–gene links or performance on new drugs.

Drug IDs and gene IDs have separate namespaces. Drug properties are aggregated only when consistent; conflicting properties become unknown and are audited. Interaction scores, publications and interaction types are **pair features**. Zero-imputed unknowns are restored using the upstream missingness flags. Both scalers are fitted using training pairs (and their incident nodes) only. No `score_*` components or target values enter the feature matrix.


In [2]:
manifest_path = INPUT_DIR / 'provenance.json'
upstream = json.loads(manifest_path.read_text(encoding='utf-8'))
if upstream.get('status') != 'completed':
    raise RuntimeError(f'Notebook 02 must finish successfully first; status={upstream.get("status")!r}')
if upstream.get('contrast') != CONTRAST:
    raise ValueError('Upstream contrast does not match configuration.')
if not upstream.get('identity_handling'):
    raise RuntimeError('Rerun the updated notebook 02 before this notebook.')
if hashlib.sha256(EDGE_FILE.read_bytes()).hexdigest() != upstream.get('dgi_edges_sha256'):
    raise ValueError('Notebook 02 input hash does not match its completed manifest.')
edges = pd.read_csv(EDGE_FILE)
if 'contrast' not in edges or not edges.contrast.eq(CONTRAST).all():
    raise ValueError('Every input association must carry the selected contrast.')
data = gnn.prepare_data(edges, split_seed=SPLIT_SEED)
for key in ['x', 'pair', 'edge_index', 'src', 'dst', 'y']:
    data[key] = data[key].to(DEVICE)
provenance = dict(status='prepared', started_utc=datetime.now(timezone.utc).isoformat(),
                  contrast=CONTRAST, task=gnn.SCOPE, input_file=str(EDGE_FILE),
                  input_sha256=hashlib.sha256(EDGE_FILE.read_bytes()).hexdigest(),
                  upstream=upstream, split_seed=SPLIT_SEED, split_unit='gene × candidate drug alias group',
                  initialization_seeds=list(INITIALIZATION_SEEDS), training=TRAINING,
                  adjacency_scope=data['adjacency_scope'],
                  selection='Minimum mean validation MSE across initialization seeds; GNN architectures only',
                  node_features=data['node_feature_names'], pair_features=data['pair_feature_names'],
                  versions=dict(python=platform.python_version(), torch=torch.__version__, pandas=pd.__version__),
                  device=str(DEVICE),
                  limitations=['Constructed target with overlapping feature ingredients; no independent activity labels',
                               'Known graph; not novel-link prediction or cold-start performance',
                               'One sample per tissue group in the upstream single-cell contrast'])
print('Associations:', len(edges), '| nodes:', len(data['node_keys']))
print({k: len(data[k]) for k in ['train', 'val', 'test']})
display(data['node_feature_audit'])
display(data['drug_identity_audit'].query('n_ids_in_alias_group > 1').head(20))


Associations: 27452 | nodes: 14047
{'train': 19207, 'val': 4127, 'test': 4118}


,node_key,feature,status


,drug,drug_id,drug_alias_group,n_ids_in_alias_group,identity_review_status
5,BHQ-880,chembl:CHEMBL2109381,alias:00213c78fde17f8511b2d11810d7d8d402240e32...,2,candidate_aliases_require_review
6,BHQ-880,hemonc:57,alias:00213c78fde17f8511b2d11810d7d8d402240e32...,2,candidate_aliases_require_review
8,RALINEPAG,chembl:CHEMBL3301604,alias:002f2d858d4037f396a584d498a0e55ed02041c4...,2,candidate_aliases_require_review
9,RALINEPAG,ncit:C152148,alias:002f2d858d4037f396a584d498a0e55ed02041c4...,2,candidate_aliases_require_review
12,VACTOSERTIB,iuphar.ligand:8107,alias:003aed8cf5b4d0d92d664b51829b01a3b37df305...,2,candidate_aliases_require_review
13,VACTOSERTIB,ncit:C116357,alias:003aed8cf5b4d0d92d664b51829b01a3b37df305...,2,candidate_aliases_require_review
43,PILARALISIB,chembl:CHEMBL3360203,alias:00fc6cfb476819a4253765669f928415284a2b63...,3,candidate_aliases_require_review
44,PILARALISIB,iuphar.ligand:7963,alias:00fc6cfb476819a4253765669f928415284a2b63...,3,candidate_aliases_require_review
45,PILARALISIB,ncit:C71705,alias:00fc6cfb476819a4253765669f928415284a2b63...,3,candidate_aliases_require_review
50,ZATOLMILAST,chembl:CHEMBL4541964,alias:0126c94a2d844c45d70fcffe0f0935b08553bb9a...,2,candidate_aliases_require_review


## Train and select using validation only

Compare GCN, GAT and GraphSAGE using the same split and repeated initialization seeds. Freeze the GNN choice using mean validation MSE **before** examining test performance. The MLP uses the same node and pair information without message passing. The selected architecture is also trained with all neighbor connections removed (self transforms retained), using the same settings and seeds.

Test comparators: selected GNN ensemble, MLP ensemble, no-neighbor ensemble, Ridge (fixed alpha=1; predictions clipped to [0,1]), and the training-label mean. The original composite score defines the labels: comparing it with itself would trivially give zero error and would not validate biology. Random initialization repeats assess computational stability; they do not replace independent cohorts or repeated patient sampling.

After choosing the GNN architecture, compare all learned approximators on validation data, including Ridge and the mean baseline, before evaluating test results. Export the original composite ranking as the primary prioritisation output. A learned approximation ranking remains experimental. Score-band test metrics expose errors in rare high-score associations.


In [3]:
TABLES_DIR.mkdir(parents=True, exist_ok=True)
provenance['status'] = 'running'
(TABLES_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
try:
    experiment = gnn.run_experiment(data, seeds=INITIALIZATION_SEEDS, **TRAINING)
except Exception as exc:
    provenance.update(status='failed_training', error=str(exc))
    (TABLES_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
    raise
display(experiment['validation'].groupby('model').validation_mse.agg(['mean', 'std']))
print('Validation-selected GNN:', experiment['selected'])
display(experiment['test_metrics'])

display(experiment['validation_comparison'])
print('Validation-preferred learned approximator:', experiment['recommended_approximator'])
print('Primary prioritisation remains the original composite score; learned fits do not validate efficacy.')


GCN: mean validation MSE = 0.001335
GAT: mean validation MSE = 0.000341
GraphSAGE: mean validation MSE = 0.000053
MLP: mean validation MSE = 0.000081
Validation-selected GNN: GraphSAGE


,mean,std
model,,
GAT,0.000341,0.000228
GCN,0.001335,0.000069
GraphSAGE,0.000053,0.000020
MLP,0.000081,0.000012


Validation-selected GNN: GraphSAGE


,model,mse,mae,r2
0,GraphSAGE,4.795591e-05,0.005106,0.991133
1,MLP,7.934219e-05,0.006662,0.985329
2,No_graph,6.833301e-05,0.006148,0.987365
3,Ridge,1.179944e-11,0.000003,1.000000
4,Training_mean,5.413971e-03,0.067590,-0.001087


,model,validation_mse
0,GraphSAGE,4.767312e-05
1,MLP,7.818634e-05
2,No_graph,6.715022e-05
3,Ridge,1.251382e-11
4,Training_mean,5.637446e-03


Validation-preferred learned approximator: Ridge
Primary prioritisation remains the original composite score; learned fits do not validate efficacy.


## Examine rankings and stability

Each row ranks a **known association**, not a distinct drug. Preserve upstream evidence, directionality, missingness, contrast and identifiers. Predictions are not calibrated probabilities. A score change relative to the constructed score is a model approximation, not a newly discovered therapeutic effect. Review disagreement and baseline performance before using the learned ranking.

The standard deviation below reflects initialization variation only; it is not clinical uncertainty. The split ledger records drugs without training labels; their graph connections remain known, so they are not truly unseen drugs.


In [4]:
ranking = experiment['ranking']
display(ranking[['rank', 'gene', 'drug', 'composite_score', 'gnn_score',
                 'initialization_score_sd', 'delta_from_constructed_score', 'label_split']].head(20))
display(experiment['stability'])
display(data['edges'].groupby('label_split').drug_seen_in_training.agg(['size', 'mean']))

display(experiment['primary_ranking'][['evidence_rank', 'gene', 'drug', 'drug_id', 'composite_score']].head(20))
display(experiment['score_band_metrics'])


,rank,gene,drug,composite_score,gnn_score,initialization_score_sd,delta_from_constructed_score,label_split
0,1,TPP1,CERLIPONASE ALFA,0.843178,0.706863,0.016613,-0.136315,train
1,2,SLC40A1,LY2928057,0.691373,0.668619,0.011078,-0.022754,train
2,3,IFNGR2,INTERFERON GAMMA-1B,0.629784,0.608974,0.010199,-0.020810,train
3,4,NEU3,OSELTAMIVIR PHOSPHATE,0.620001,0.607790,0.011476,-0.012211,train
4,5,HPD,NITISINONE,0.623160,0.601849,0.007480,-0.021312,train
5,6,TCN1,COBALAMIN,0.502527,0.513610,0.007967,0.011082,test
6,7,PROM1,ICT-121 DENDRITIC CELL VACCINE,0.494519,0.505306,0.011939,0.010787,validation
7,8,RPA1,CHEMBL:CHEMBL1765371,0.497044,0.503267,0.012321,0.006223,train
8,9,UQCR10,FAMOXADONE,0.493796,0.498324,0.012381,0.004528,test
9,10,ELN,VONAPANITASE,0.489536,0.497571,0.010486,0.008035,train


,seed_a,seed_b,spearman,top20_overlap
0,42,43,0.998256,1.0
1,42,44,0.997278,1.0
2,43,44,0.998078,1.0


,size,mean
label_split,,
test,4118,0.654930
train,19207,1.000000
validation,4127,0.662224


,evidence_rank,gene,drug,drug_id,composite_score
0,1,TPP1,CERLIPONASE ALFA,rxcui:1922436,0.843178
1,2,SLC40A1,LY2928057,iuphar.ligand:8416,0.691373
2,3,IFNGR2,INTERFERON GAMMA-1B,rxcui:5882,0.629784
3,4,HPD,NITISINONE,rxcui:61805,0.623160
4,5,NEU3,OSELTAMIVIR PHOSPHATE,rxcui:259275,0.620001
5,6,TCN1,COBALAMIN,ncit:C939,0.502527
6,7,RPA1,CHEMBL:CHEMBL1765371,chembl:CHEMBL1765371,0.497044
7,8,PROM1,ICT-121 DENDRITIC CELL VACCINE,ncit:C124652,0.494519
8,9,UQCR10,FAMOXADONE,chembl:CHEMBL10118,0.493796
9,10,ELN,VONAPANITASE,chembl:CHEMBL3545070,0.489536


,score_band,n_pairs,model,mse,mae,r2
0,0–0.1,2478,GraphSAGE,3.981785e-05,0.005065,0.832656
1,0–0.1,2478,MLP,6.175278e-05,0.006392,0.740470
2,0–0.1,2478,No_graph,5.300021e-05,0.005932,0.777254
3,0–0.1,2478,Ridge,5.164786e-12,0.000002,1.000000
4,0–0.1,2478,Training_mean,3.562477e-03,0.057659,-13.972140
5,0.1–0.25,1536,GraphSAGE,5.587695e-05,0.004911,0.953064
6,0.1–0.25,1536,MLP,9.899884e-05,0.006703,0.916842
7,0.1–0.25,1536,No_graph,8.303818e-05,0.006036,0.930249
8,0.1–0.25,1536,Ridge,1.712985e-11,0.000004,1.000000
9,0.1–0.25,1536,Training_mean,6.927489e-03,0.077413,-4.819017


## Export reproducible artifacts and the report

Save selected architecture checkpoints with their actual names and initialization seeds, MLP/no-neighbor checkpoints, fitted preprocessing and Ridge baseline, graph tensors, node identities, split ledger, all evidence fields, per-initialization predictions, and input provenance. Existing historical top-level outputs are retained as historical artifacts; use this run's paths below.


In [5]:
provenance.update(selected_gnn=experiment['selected'], recommended_approximator=experiment['recommended_approximator'], primary_ranking='original composite evidence score', status='exporting')
try:
    gnn.export_experiment(experiment, data, TABLES_DIR, MODELS_DIR, provenance)
    provenance.update(status='completed', completed_utc=datetime.now(timezone.utc).isoformat())
    report_path = gnn.generate_outputs(experiment, data, FIGURES_DIR, REPORTS_DIR, provenance)
except Exception as exc:
    provenance.update(status='failed_export', error=str(exc))
    raise
finally:
    (TABLES_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
print('Ranking:', TABLES_DIR / 'gnn_drug_ranking.csv')
print('Models:', MODELS_DIR)
print('Report:', report_path)

print('Primary evidence ranking:', TABLES_DIR / 'composite_drug_ranking.csv')
print('Biological review queue:', TABLES_DIR / 'biological_review_queue.csv')


Ranking: c:\Users\shoko\OneDrive\Desktop\project\HCC_DD\results\tables\evidence_score_regression\pooled\gnn_drug_ranking.csv
Models: c:\Users\shoko\OneDrive\Desktop\project\HCC_DD\models\evidence_score_regression\pooled
Report: c:\Users\shoko\OneDrive\Desktop\project\HCC_DD\results\reports\evidence_score_regression\pooled\03_gnn_drug_ranking_report.html
Primary evidence ranking: c:\Users\shoko\OneDrive\Desktop\project\HCC_DD\results\tables\evidence_score_regression\pooled\composite_drug_ranking.csv
Biological review queue: c:\Users\shoko\OneDrive\Desktop\project\HCC_DD\results\tables\evidence_score_regression\pooled\biological_review_queue.csv
